In [ ]:
import pandas as pd

#clean data and remove any rows with missing values

#check if the file exists and it loads correctly
raw = pd.read_csv("Test1.csv")
print(f"Row and Column types: {raw.shape}")
print(f"number of rows and columns: {raw.shape[0]} rows, {raw.shape[1]} columns")

#check for duplicates in the data and remove them
print(f"Duplicates: {raw.duplicated().sum()}")
clean = raw.drop_duplicates()

key = [
    "Reference area", "TIME_PERIOD", "Financing scheme",
    "Price base", "Health function", "Healthcare provider",
    "Unit of measure",
]
print(f"Duplicates in key columns: {clean.duplicated(key).sum()}")

#checks for spending values that are missing or less than or equal to 0 and checks for time periods that are outside the range of 1970 to 2025
print(f"Missing spending values: {clean['OBS_VALUE'].isna().sum()}")
print(f"Spending values <= 0: { (clean['OBS_VALUE'] <= 0).sum() }")
print(f"Time periods outside 1970-2025: {clean.loc[~clean['TIME_PERIOD'].between(1970, 2025), 'TIME_PERIOD'].unique() }") #checks if years are in this range(Can be removed if the data is for a different time period)

#In this dataset, there are some columns that seem like duplicates. Like PRICE_BASE and Price base
# However, these arent duplicates. Instead, one column holds a key for the other column.
#For example, OBS_STATUS has a value called "E" which is a key for the value "Estimated" in the Observation status column.
pairs = [
    ("FINANCING_SCHEME", "Financing scheme"),
    ("PRICE_BASE", "Price base"),
    ("OBS_STATUS", "Observation status"),
    ("REF_AREA", "Reference area"),
    ("UNIT_MEASURE", "Unit of measure"),
]



#this for loop goes through the pairs of columns and checks if the number of unique values in the key column is
#equal to the number of unique values in the value column. If they are not equal, it prints the maximum number of unique
#values in the value column for each unique value in the key column. This is to check if there are any missing or extra 
#values in the value column that do not have a corresponding key in the key column.
#if they are equal, it prints the maximum number of unique values in the value column for each unique value in the key column.
for code, label in pairs:
    labels_per_code = clean.groupby(code, dropna=False)[label].nunique(dropna=False)
    print(code, labels_per_code.max())



print(clean["Observation status"].isna().sum())  #finds blanks in the Observation status column and prints the number of blanks
print(clean.loc[clean["Observation status"].notna(),
                ["TIME_PERIOD", "Financing scheme", "Price base", "OBS_VALUE", "Observation status"]]) 
#finds columns that arent blank in this row and shows the values of the 5 columns
# This is because these are estimated values and not actual values 


#checks if total prices is correct and if it equals Government/compulsory schemes + Voluntary schemes/household out-of-pocket payments
const = clean[clean["Price base"] == "Constant prices"]
wide = const.pivot(index="TIME_PERIOD", columns="Financing scheme", values="OBS_VALUE")
gap = (
    wide["Government/compulsory schemes"]
    + wide["Voluntary schemes/household out-of-pocket payments"]
    - wide["Total"]
).abs()
print(f"Maximum gap: {gap.max()}")



Row and Column types: (1052, 46)
number of rows and columns: 1052 rows, 46 columns
Duplicates: 0
Duplicates in key columns: 0
Missing spending values: 0
Spending values <= 0: 0
Time periods outside 1970-2025: []
FINANCING_SCHEME 1
PRICE_BASE 1
OBS_STATUS 1
REF_AREA 1
UNIT_MEASURE 1
1046
     TIME_PERIOD                                   Financing scheme  \
801         2025  Voluntary schemes/household out-of-pocket paym...   
802         2025                                              Total   
803         2025                      Government/compulsory schemes   
804         2025  Voluntary schemes/household out-of-pocket paym...   
805         2025                                              Total   
806         2025                      Government/compulsory schemes   

          Price base  OBS_VALUE Observation status  
801   Current prices   2662.472    Estimated value  
802  Constant prices  13049.083    Estimated value  
803   Current prices  13180.807    Estimated value  
80